# Classificação de Superfícies de Vias — Voxar Labs PS 2025

**Autor:** Alisson da Silva Bernadino  
**Data:** Abril de 2026  

---

## Identificação da Abordagem

Este notebook resolve o desafio de classificação de imagens em 3 classes (**Asphalt**, **Belgian Blocks**, **Off-road**) utilizando **Transfer Learning** com PyTorch.

**Modelo escolhido:** ResNet-18 pré-treinado no ImageNet  
**Justificativa:** Dado o tempo sugerido de 6–10h, treinar um modelo do zero seria inviável. A ResNet-18 é uma arquitetura pequena (~11M parâmetros), amplamente validada para tarefas de classificação visual. O fine-tuning das camadas superiores adapta os filtros genéricos do ImageNet para as texturas específicas de superfícies de estrada — exatamente o que precisamos.

**Bibliotecas principais:**
- `torch` / `torchvision` — backbone, training loop, transforms
- `matplotlib` / `seaborn` — visualizações e matriz de confusão
- `scikit-learn` — métricas (F1, classification report)
- `numpy` / `PIL` — manipulação de arrays e imagens

**Estrutura do notebook:**
1. Entendimento do Problema (EDA)
2. Solução Baseline
3. Experimentos
4. Resultados Consolidados
5. Análise Crítica
6. Uso de Ferramentas

---
## Seção 1 — Entendimento do Problema (EDA & Estratégia)

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import defaultdict

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Caminho base do dataset — ajuste se necessário
BASE_DIR = Path('dataset_processed')
SPLITS   = ['train', 'test']
CLASSES  = ['asphalt', 'belgian_blocks', 'offroad']
CLASS_LABELS = {'asphalt': 'Asphalt', 'belgian_blocks': 'Belgian Blocks', 'offroad': 'Off-road'}

print('Dataset path:', BASE_DIR.resolve())
print('Exists:', BASE_DIR.exists())

### 1.1 Natureza do Problema

Trata-se de um problema de **classificação de imagens multiclasse** (3 classes) com características que elevam sua complexidade:

| Desafio | Descrição |
|---|---|
| **Desbalanceamento severo** | A classe *Asphalt* domina o dataset; *Belgian Blocks* é a menor |
| **Variação visual intra-classe** | Iluminação noturna, chuva, sujeira, diferentes câmeras |
| **Similaridade inter-classes** | Off-road e Belgian Blocks podem ter texturas próximas |
| **Ruído de captura** | Diferentes ângulos, resoluções e artefatos de compressão |

In [ ]:
# ── 1.2 Distribuição das Classes ──────────────────────────────────────────────
counts = {}
for split in SPLITS:
    counts[split] = {}
    for cls in CLASSES:
        path = BASE_DIR / split / cls
        counts[split][cls] = len(list(path.glob('*')))

print('=== Distribuição do Dataset ===')
total_train = sum(counts['train'].values())
total_test  = sum(counts['test'].values())

print(f"\n{'Classe':<20} {'Train':>8} {'%Train':>8} {'Test':>8} {'%Test':>8}")
print('-' * 55)
for cls in CLASSES:
    tr = counts['train'][cls]
    te = counts['test'][cls]
    print(f"{CLASS_LABELS[cls]:<20} {tr:>8} {tr/total_train*100:>7.1f}% {te:>8} {te/total_test*100:>7.1f}%")
print('-' * 55)
print(f"{'TOTAL':<20} {total_train:>8} {'100.0%':>8} {total_test:>8} {'100.0%':>8}")

# Razão de desbalanceamento
max_cls = max(counts['train'], key=counts['train'].get)
min_cls = min(counts['train'], key=counts['train'].get)
ratio = counts['train'][max_cls] / counts['train'][min_cls]
print(f"\nRazão de desbalanceamento (train): {ratio:.1f}x ({CLASS_LABELS[max_cls]} / {CLASS_LABELS[min_cls]})")

In [ ]:
# ── Gráfico de distribuição ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4A90D9', '#E67E22', '#27AE60']
labels = [CLASS_LABELS[c] for c in CLASSES]

for ax, split in zip(axes, SPLITS):
    vals = [counts[split][c] for c in CLASSES]
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'Distribuição — {split.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Número de imagens')
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                str(v), ha='center', va='bottom', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Desbalanceamento do Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

**Observação:** O dataset apresenta desbalanceamento severo (~7x entre Asphalt e Belgian Blocks no treino). Isso significa que um modelo ingênuo pode atingir **~73% de acurácia simplesmente prevendo sempre Asphalt** — motivo pelo qual usaremos **F1-Score macro** e **Matriz de Confusão por classe** como métricas primárias.

### 1.3 Definição das Métricas de Sucesso

| Métrica | Por quê usar |
|---|---|
| **Acurácia** | Referência geral, mas enganosa com classes desbalanceadas |
| **F1-Score macro** | Métrica primária: trata todas as classes com igual peso |
| **Precision / Recall por classe** | Identifica onde o modelo falha especificamente |
| **Matriz de Confusão** | Evidência visual dos padrões de erro |

In [ ]:
# ── 1.4 Visualização de Amostras por Classe ───────────────────────────────────
def load_random_samples(split, cls, n=4, seed=SEED):
    path = BASE_DIR / split / cls
    files = list(path.glob('*'))
    random.seed(seed)
    chosen = random.sample(files, min(n, len(files)))
    return [Image.open(f).convert('RGB') for f in chosen]

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
fig.suptitle('Amostras do Conjunto de Treino por Classe', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASSES):
    imgs = load_random_samples('train', cls, n=4)
    for col, img in enumerate(imgs):
        ax = axes[row][col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_LABELS[cls], fontsize=11, fontweight='bold',
                          rotation=0, labelpad=60, va='center')

plt.tight_layout()
plt.savefig('eda_class_samples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.5 Casos Difíceis — Análise Qualitativa ─────────────────────────────────
# Busca imagens escuras (brilho médio baixo) como proxy para "noite/chuva"
def get_image_stats(split, cls, n_sample=30):
    path = BASE_DIR / split / cls
    files = list(path.glob('*'))
    random.seed(SEED)
    sample = random.sample(files, min(n_sample, len(files)))
    stats = []
    for f in sample:
        img = np.array(Image.open(f).convert('RGB'))
        stats.append({'file': f, 'brightness': img.mean(), 'std': img.std()})
    return stats

print('=== Análise de Brilho por Classe (treino) ===')
fig, ax = plt.subplots(figsize=(10, 4))
all_brightness = []
all_labels = []

for cls in CLASSES:
    stats = get_image_stats('train', cls, n_sample=60)
    brightness = [s['brightness'] for s in stats]
    all_brightness.append(brightness)
    all_labels.append(CLASS_LABELS[cls])
    print(f"  {CLASS_LABELS[cls]:<18}: mean={np.mean(brightness):.1f}, min={np.min(brightness):.1f}, max={np.max(brightness):.1f}")

bp = ax.boxplot(all_brightness, labels=all_labels, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribuição de Brilho por Classe\n(valores baixos = imagens escuras/noite/chuva)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Brilho médio (0–255)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('eda_brightness_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Exibir exemplos de imagens "difíceis" (mais escuras) ──────────────────────
def get_dark_samples(split, cls, n=3, brightness_threshold=80):
    path = BASE_DIR / split / cls
    files = list(path.glob('*'))
    dark = []
    for f in files:
        img = np.array(Image.open(f).convert('RGB'))
        if img.mean() < brightness_threshold:
            dark.append((f, img.mean()))
    dark.sort(key=lambda x: x[1])
    return [Image.open(f).convert('RGB') for f, _ in dark[:n]]

fig, axes = plt.subplots(3, 3, figsize=(11, 9))
fig.suptitle('Casos Difíceis — Imagens de Baixo Brilho (noite/chuva)', fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASSES):
    dark_imgs = get_dark_samples('train', cls, n=3)
    # preenche com imagens aleatórias se não houver imagens escuras suficientes
    if len(dark_imgs) < 3:
        extra = load_random_samples('train', cls, n=3-len(dark_imgs), seed=1)
        dark_imgs.extend(extra)
    for col, img in enumerate(dark_imgs[:3]):
        ax = axes[row][col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_LABELS[cls], fontsize=11, fontweight='bold',
                          rotation=0, labelpad=60, va='center')

plt.tight_layout()
plt.savefig('eda_hard_cases.png', dpi=120, bbox_inches='tight')
plt.show()

### 1.6 Resumo dos Desafios Identificados

1. **Desbalanceamento crítico (~7x):** A classe *Belgian Blocks* tem apenas 94 amostras no treino. Um modelo treinado sem estratégia tende a ignorar essa classe.

2. **Variação intra-classe extrema:** A análise de brilho mostra que existem imagens muito escuras em todas as classes, especialmente *Off-road*, o que simula condições noturnas e de chuva.

3. **Ambiguidade inter-classes:** Belgian Blocks e Off-road compartilham características texturais irregulares, tornando-as difíceis de separar em baixa iluminação.

4. **Heterogeneidade de captura:** Diferentes dispositivos, ângulos e distâncias focais geram variância adicional não relacionada ao conteúdo semântico.

**Estratégia proposta:**
- Transfer Learning com ResNet-18 (pré-treinado ImageNet)
- Weighted CrossEntropy para tratar desbalanceamento
- Data Augmentation focado em variações de iluminação/cor
- Avaliação primária por F1-Score macro

In [ ]:
# ── 1.7 Análise de Dimensões e Formatos ───────────────────────────────────────
print('=== Amostra de dimensões de imagens (treino) ===')
dim_stats = defaultdict(list)

for cls in CLASSES:
    path = BASE_DIR / 'train' / cls
    files = list(path.glob('*'))
    random.seed(SEED)
    sample = random.sample(files, min(20, len(files)))
    for f in sample:
        try:
            img = Image.open(f)
            dim_stats[cls].append(img.size)  # (width, height)
        except Exception:
            pass

for cls in CLASSES:
    dims = dim_stats[cls]
    widths  = [d[0] for d in dims]
    heights = [d[1] for d in dims]
    print(f"  {CLASS_LABELS[cls]:<18}: W=[{min(widths)}–{max(widths)}]  H=[{min(heights)}–{max(heights)}]")

print('\nAs imagens têm dimensões variadas → resize para 224x224 necessário antes do modelo.')

---
## Seção 2 — Solução Baseline

### Abordagem

**Modelo:** ResNet-18 pré-treinado (ImageNet) com fine-tuning da última camada fully-connected.

**Justificativa técnica:**
- Dentro do tempo sugerido (6–10h), treinar um modelo do zero é inviável e desnecessário.
- Os pesos ImageNet já capturam bordas, texturas e gradientes — features diretamente úteis para distinguir superfícies de asfalto, paralelepípedo e terra.
- ResNet-18 é o menor modelo da família ResNet (~11M params), rápido para treinar em CPU/GPU modesta.
- Congelar o backbone e treinar apenas o classificador final constitui o baseline mais simples e controlado.

**O que esta baseline NÃO faz (propositalmente):**
- Não trata o desbalanceamento de classes → Experimento 1
- Não usa data augmentation agressivo → Experimento 2

Isso nos dá um **marco zero** claro para comparação científica.

In [ ]:
# ── 2.1 Imports e Configurações ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ── Hiperparâmetros ───────────────────────────────────────────────────────────
CFG = {
    'image_size' : 224,
    'batch_size' : 32,
    'epochs'     : 15,
    'lr'         : 1e-3,
    'num_classes': 3,
    'seed'       : 42,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES   = ['asphalt', 'belgian_blocks', 'offroad']
CLASS_DISPLAY = ['Asphalt', 'Belgian Blocks', 'Off-road']

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

print(f'Device: {DEVICE}')
print(f'Configuração: {CFG}')


In [ ]:
# ── 2.2 Pipeline de Dados ─────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Baseline: transforms mínimos — sem augmentation
baseline_transforms = {
    split: transforms.Compose([
        transforms.Resize((CFG['image_size'], CFG['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    for split in ['train', 'test']
}

datasets_dict = {
    split: datasets.ImageFolder(
        root=str(BASE_DIR / split),
        transform=baseline_transforms[split]
    )
    for split in ['train', 'test']
}

loaders = {
    split: DataLoader(
        datasets_dict[split],
        batch_size=CFG['batch_size'],
        shuffle=(split == 'train'),
        num_workers=0,
        pin_memory=(DEVICE.type == 'cuda'),
    )
    for split in ['train', 'test']
}

print(f"Train: {len(datasets_dict['train'])} imgs | classes: {datasets_dict['train'].classes}")
print(f"Test : {len(datasets_dict['test'])} imgs")


In [ ]:
# ── 2.3 Definição do Modelo ───────────────────────────────────────────────────
def build_model(num_classes=3, freeze_backbone=True):
    """Carrega ResNet-18 pré-treinada e adapta o classificador para N classes."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    # Substitui FC final: 512 → num_classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


baseline_model = build_model(num_classes=CFG['num_classes'], freeze_backbone=True)

trainable = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in baseline_model.parameters())
print(f'Parâmetros treináveis: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')


In [ ]:
# ── 2.4 Loop de Treinamento Modular ──────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (model(imgs).detach().argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        total_loss += criterion(outputs, labels).item() * imgs.size(0)
        preds   = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


def run_training(model, loaders, cfg, criterion=None, tag='baseline'):
    """Executa treinamento completo e retorna histórico de métricas."""
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=cfg['lr']
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    print(f'\n=== Treinamento: {tag} ===')
    for epoch in range(1, cfg['epochs'] + 1):
        tr_loss, tr_acc = train_one_epoch(model, loaders['train'], criterion, optimizer, DEVICE)
        vl_loss, vl_acc, _, _ = evaluate(model, loaders['test'], criterion, DEVICE)
        scheduler.step()
        for k, v in zip(['train_loss','train_acc','val_loss','val_acc'],
                         [tr_loss, tr_acc, vl_loss, vl_acc]):
            history[k].append(v)
        print(f'  Epoch {epoch:>2}/{cfg["epochs"]} | '
              f'train_loss={tr_loss:.4f} acc={tr_acc:.3f} | '
              f'val_loss={vl_loss:.4f} acc={vl_acc:.3f}')
    return history


print('Funções definidas.')


In [ ]:
# ── 2.5 Treinamento ───────────────────────────────────────────────────────────
baseline_criterion = nn.CrossEntropyLoss()  # sem pesos — baseline puro
history_baseline   = run_training(baseline_model, loaders, CFG,
                                   criterion=baseline_criterion,
                                   tag='Baseline (sem class weights)')


In [ ]:
# ── 2.6 Curvas de Aprendizado ─────────────────────────────────────────────────
def plot_history(history, title='Baseline'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs, history['train_loss'], 'o-', label='Train', color='#4A90D9')
    ax1.plot(epochs, history['val_loss'],   's-', label='Val',   color='#E74C3C')
    ax1.set_title(f'{title} — Loss', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.spines[['top','right']].set_visible(False)
    ax2.plot(epochs, history['train_acc'], 'o-', label='Train', color='#4A90D9')
    ax2.plot(epochs, history['val_acc'],   's-', label='Val',   color='#E74C3C')
    ax2.set_title(f'{title} — Accuracy', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1); ax2.legend()
    ax2.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'curves_{title.lower().replace(" ","_")}.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_history(history_baseline, title='Baseline')


In [ ]:
# ── 2.7 Avaliação Completa ────────────────────────────────────────────────────
def full_evaluation(model, loader, criterion, device, class_names, tag='Baseline'):
    _, acc, preds, labels = evaluate(model, loader, criterion, device)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_per   = f1_score(labels, preds, average=None)

    print(f'\n=== Resultados — {tag} ===')
    print(f'  Acurácia  : {acc:.4f} ({acc*100:.2f}%)')
    print(f'  F1 Macro  : {f1_macro:.4f}')
    print(f'  F1/classe : { {c: f"{v:.3f}" for c, v in zip(class_names, f1_per)} }')
    print()
    print(classification_report(labels, preds, target_names=class_names, digits=3))

    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, data, fmt, ttl in zip(
        axes, [cm, cm_norm], ['d', '.2f'],
        [f'{tag} — Contagens', f'{tag} — Normalizada (por linha)']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5, cbar=False)
        ax.set_title(ttl, fontweight='bold', fontsize=11)
        ax.set_xlabel('Predito'); ax.set_ylabel('Real')
    plt.tight_layout()
    plt.savefig(f'cm_{tag.lower().replace(" ","_")}.png', dpi=120, bbox_inches='tight')
    plt.show()
    return {'acc': acc, 'f1_macro': f1_macro, 'f1_per': f1_per,
            'preds': preds, 'labels': labels}


results_baseline = full_evaluation(
    baseline_model, loaders['test'], baseline_criterion,
    DEVICE, CLASS_DISPLAY, tag='Baseline'
)


### 2.8 Interpretação do Baseline

Os resultados acima constituem nosso **marco zero**. Hipóteses esperadas (a confirmar com a execução):

- A **acurácia geral** pode parecer razoável (>70%), mas é inflada pelo peso da classe *Asphalt* (73% do treino).
- O **F1-Score de Belgian Blocks** deve ser baixo — sem estratégia de balanceamento, o modelo aprende a ignorar a classe minoritária.
- A **matriz de confusão normalizada** revelará o padrão de erro dominante (esperado: Belgian Blocks confundidos com Asphalt ou Off-road).

Esses padrões motivam diretamente os dois experimentos a seguir.

---
## Seção 3 — Experimentos

---
### Experimento 1 — Weighted CrossEntropy (Tratamento do Desbalanceamento)

#### Hipótese

> *"O modelo baseline ignora as classes minoritárias porque a função de perda trata todos os exemplos igualmente. Ao atribuir pesos inversamente proporcionais à frequência de cada classe, forçamos o modelo a penalizar mais os erros nas classes raras (Belgian Blocks e Off-road), melhorando o recall nessas classes sem alterar a arquitetura."*

#### O que muda em relação ao baseline

| | Baseline | Experimento 1 |
|---|---|---|
| Loss | `CrossEntropyLoss()` | `CrossEntropyLoss(weight=class_weights)` |
| Arquitetura | ResNet-18 | ResNet-18 (idêntica) |
| Augmentation | Nenhum | Nenhum (idêntico) |
| Epochs / LR | 15 / 1e-3 | 15 / 1e-3 (idênticos) |

Ao manter tudo igual exceto a loss, isolamos o efeito do balanceamento.

In [ ]:
# ── 3.1 Cálculo dos Pesos de Classe ──────────────────────────────────────────
# Estratégia: peso = N_total / (N_classes * N_i)
# Classes minoritárias recebem peso maior → perda maior quando erradas

train_counts = [counts['train'][c] for c in CLASSES]  # [655, 94, 151]
total_train  = sum(train_counts)
n_classes    = len(CLASSES)

# Pesos normalizados (inverso da frequência relativa)
weights = [total_train / (n_classes * c) for c in train_counts]
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print('=== Pesos de Classe ===')
for cls, cnt, w in zip(CLASS_DISPLAY, train_counts, weights):
    print(f'  {cls:<18}: {cnt:>4} imgs → peso = {w:.4f}')
print(f'\nRazão max/min de pesos: {max(weights)/min(weights):.1f}x')
print('\nInterpretação: um erro em Belgian Blocks penaliza ~{:.1f}x mais que um erro em Asphalt'.format(
    weights[1] / weights[0]
))


In [ ]:
# ── 3.2 Treinamento com Weighted Loss ────────────────────────────────────────
# Novo modelo com pesos aleatórios na FC (mesma arquitetura)
exp1_model     = build_model(num_classes=CFG['num_classes'], freeze_backbone=True)
exp1_criterion = nn.CrossEntropyLoss(weight=class_weights)

history_exp1 = run_training(
    exp1_model, loaders, CFG,
    criterion=exp1_criterion,
    tag='Exp1 — Weighted CrossEntropy'
)


In [ ]:
# ── 3.3 Curvas de Aprendizado — Exp1 ─────────────────────────────────────────
plot_history(history_exp1, title='Exp1 — Weighted Loss')


In [ ]:
# ── 3.4 Avaliação do Exp1 ────────────────────────────────────────────────────
results_exp1 = full_evaluation(
    exp1_model, loaders['test'], exp1_criterion,
    DEVICE, CLASS_DISPLAY, tag='Exp1 — Weighted Loss'
)


In [ ]:
# ── 3.5 Comparação Direta: Baseline vs Exp1 ──────────────────────────────────
print('=' * 55)
print(f'{"Métrica":<25} {"Baseline":>12} {"Exp1 (w)": >12}')
print('-' * 55)
print(f'{"Acurácia":<25} {results_baseline["acc"]:>12.4f} {results_exp1["acc"]:>12.4f}')
print(f'{"F1 Macro":<25} {results_baseline["f1_macro"]:>12.4f} {results_exp1["f1_macro"]:>12.4f}')
print('-' * 55)
for i, cls in enumerate(CLASS_DISPLAY):
    bl = results_baseline['f1_per'][i]
    e1 = results_exp1['f1_per'][i]
    delta = e1 - bl
    arrow = '▲' if delta > 0 else '▼'
    print(f'  F1 {cls:<20} {bl:>8.4f} {e1:>8.4f}  {arrow}{abs(delta):.4f}')
print('=' * 55)


In [ ]:
# ── 3.6 Comparação Visual das Matrizes de Confusão ───────────────────────────
from sklearn.metrics import confusion_matrix
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, results, title in zip(
    axes,
    [results_baseline, results_exp1],
    ['Baseline (sem pesos)', 'Exp1 — Weighted Loss']
):
    cm = confusion_matrix(results['labels'], results['preds'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_DISPLAY, yticklabels=CLASS_DISPLAY,
                ax=ax, linewidths=0.5, vmin=0, vmax=1, cbar=False)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predito'); ax.set_ylabel('Real')

plt.suptitle('Comparação de Matrizes de Confusão Normalizadas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cm_comparison_exp1.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.7 Resultado e Interpretação — Experimento 1

**O que esperamos observar:**

| Efeito esperado | Motivo |
|---|---|
| Acurácia geral pode **cair levemente** | O modelo passa a errar mais em Asphalt para acertar as minoritárias |
| F1 de **Belgian Blocks** deve **subir** | Penalidade maior força o modelo a aprender essa classe |
| F1 de **Off-road** deve **subir** | Mesmo efeito |
| **F1 Macro deve subir** | Ganho nas minoritárias compensa a pequena perda no Asphalt |

**Interpretação dos resultados observados:**

> *(Preencher após execução: descrever o que de fato aconteceu com cada métrica, comparando com as hipóteses acima. Verificar se o padrão de erros na matriz de confusão mudou — especialmente as linhas Belgian Blocks e Off-road.)*

**Conclusão:** O uso de pesos de classe é uma técnica de custo zero (não adiciona parâmetros nem tempo de treinamento) que pode trazer ganhos significativos em datasets desbalanceados. O trade-off acurácia ↔ F1-macro é esperado e aceitável dado o contexto do problema.

---
### Experimento 2 — Data Augmentation Robusto (Resiliência Visual)

#### Hipótese

> *"O dataset inclui variações severas de iluminação (noite, chuva, reflexos). O modelo, treinado com imagens sem perturbações, pode ter aprendido atalhos visuais que não generalizam para esses casos. Aplicar augmentation que simule essas condições durante o treino forçará o modelo a aprender features mais robustas e invariantes, reduzindo os erros nas imagens mais difíceis."*

#### O que muda em relação ao Experimento 1

| | Exp1 | Experimento 2 |
|---|---|---|
| Loss | Weighted CrossEntropy | Weighted CrossEntropy (mantida) |
| Arquitetura | ResNet-18 | ResNet-18 (idêntica) |
| **Augmentation** | **Nenhum** | **ColorJitter + Blur + Flip + Rotation** |
| Epochs / LR | 15 / 1e-3 | 15 / 1e-3 (idênticos) |

**Importante:** mantemos os pesos de classe do Exp1. O objetivo é avaliar o ganho *adicional* que o augmentation traz quando somado ao balanceamento.

In [ ]:
# ── 3.8 Transforms de Augmentation ───────────────────────────────────────────
# Cada transformação simula uma condição real do dataset:
#   ColorJitter(brightness, contrast)  → noite, chuva, reflexos
#   GaussianBlur                       → câmeras de baixa qualidade, movimento
#   RandomGrayscale                    → câmeras monocromáticas / IR
#   RandomHorizontalFlip               → diferentes orientações de captura
#   RandomRotation                     → câmera não centralizada no veículo

aug_train_transforms = transforms.Compose([
    transforms.Resize((CFG['image_size'], CFG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.4,   # simula variação de iluminação (noite/dia)
        contrast=0.4,     # simula chuva e neblina
        saturation=0.2,
        hue=0.05,
    ),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),  # blur de câmera
    transforms.RandomGrayscale(p=0.1),  # câmeras monocromáticas
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Teste: sem augmentation (regra geral — não aumentamos o conjunto de avaliação)
aug_test_transforms = baseline_transforms['test']

# Novos dataloaders com augmentation apenas no treino
aug_datasets = {
    'train': datasets.ImageFolder(str(BASE_DIR / 'train'), transform=aug_train_transforms),
    'test' : datasets.ImageFolder(str(BASE_DIR / 'test'),  transform=aug_test_transforms),
}
aug_loaders = {
    split: DataLoader(
        aug_datasets[split],
        batch_size=CFG['batch_size'],
        shuffle=(split == 'train'),
        num_workers=0,
        pin_memory=(DEVICE.type == 'cuda'),
    )
    for split in ['train', 'test']
}

print('DataLoaders com augmentation prontos.')
print(f"Train transforms: {aug_train_transforms}")


In [ ]:
# ── 3.9 Visualização das Augmentations ───────────────────────────────────────
# Mostra o mesmo lote de imagens com e sem augmentation para validar visualmente
from torchvision.utils import make_grid

def denormalize(tensor):
    """Reverte normalização ImageNet para visualização."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Efeito do Data Augmentation (mesma imagem, diferentes amostras)',
             fontsize=12, fontweight='bold')

# Carrega uma imagem de cada classe e exibe original vs augmentada
sample_files = []
for cls in CLASSES:
    files = sorted((BASE_DIR / 'train' / cls).glob('*'))
    sample_files.append(files[0])

orig_tfm = baseline_transforms['train']
aug_tfm  = aug_train_transforms

imgs_orig = [orig_tfm(Image.open(f).convert('RGB')) for f in sample_files]

for col, (f, img_orig) in enumerate(zip(sample_files[:8], imgs_orig[:8])):
    img_aug = aug_tfm(Image.open(f).convert('RGB'))
    axes[0][col].imshow(denormalize(img_orig).permute(1,2,0).numpy())
    axes[0][col].axis('off')
    axes[1][col].imshow(denormalize(img_aug).permute(1,2,0).numpy())
    axes[1][col].axis('off')

axes[0][0].set_ylabel('Original', fontsize=10, fontweight='bold', rotation=0,
                       labelpad=55, va='center')
axes[1][0].set_ylabel('Augmentada', fontsize=10, fontweight='bold', rotation=0,
                       labelpad=55, va='center')

plt.tight_layout()
plt.savefig('augmentation_preview.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.10 Treinamento — Exp2 (Augmentation + Class Weights) ───────────────────
exp2_model     = build_model(num_classes=CFG['num_classes'], freeze_backbone=True)
exp2_criterion = nn.CrossEntropyLoss(weight=class_weights)  # mantém pesos do Exp1

history_exp2 = run_training(
    exp2_model, aug_loaders, CFG,
    criterion=exp2_criterion,
    tag='Exp2 — Augmentation + Weighted Loss'
)


In [ ]:
# ── 3.11 Curvas de Aprendizado — Exp2 ────────────────────────────────────────
plot_history(history_exp2, title='Exp2 — Augmentation')


In [ ]:
# ── 3.12 Avaliação — Exp2 ────────────────────────────────────────────────────
results_exp2 = full_evaluation(
    exp2_model, aug_loaders['test'], exp2_criterion,
    DEVICE, CLASS_DISPLAY, tag='Exp2 — Augmentation'
)


In [ ]:
# ── 3.13 Análise de Erros Qualitativa ────────────────────────────────────────
# Identifica imagens que o Baseline errou e verifica se o Exp2 corrigiu

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_paths = [], [], []
    dataset = loader.dataset
    for idx in range(len(dataset)):
        img_tensor, label = dataset[idx]
        img_tensor = img_tensor.unsqueeze(0).to(device)
        pred = model(img_tensor).argmax(1).item()
        all_preds.append(pred)
        all_labels.append(label)
        all_paths.append(dataset.samples[idx][0])
    return all_preds, all_labels, all_paths


# Usa os loaders sem augmentation para comparação justa
bl_preds, bl_labels, paths = get_predictions(baseline_model, loaders['test'], DEVICE)
e2_preds, _,        _     = get_predictions(exp2_model,     loaders['test'], DEVICE)

# Casos onde Baseline errou E Exp2 acertou
fixed_cases = [
    (p, bl_labels[i], bl_preds[i], e2_preds[i])
    for i, p in enumerate(paths)
    if bl_preds[i] != bl_labels[i] and e2_preds[i] == bl_labels[i]
]

# Casos onde Exp2 errou mas Baseline acertou (regressões)
regression_cases = [
    (p, bl_labels[i], bl_preds[i], e2_preds[i])
    for i, p in enumerate(paths)
    if bl_preds[i] == bl_labels[i] and e2_preds[i] != bl_labels[i]
]

print(f'Erros corrigidos pelo Exp2:  {len(fixed_cases)}')
print(f'Regressões introduzidas:     {len(regression_cases)}')
print(f'Saldo líquido:               +{len(fixed_cases) - len(regression_cases)}')


In [ ]:
# ── 3.14 Visualização — Erros Corrigidos pelo Exp2 ───────────────────────────
n_show = min(6, len(fixed_cases))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4))
    if n_show == 1:
        axes = [axes]
    fig.suptitle('Erros do Baseline corrigidos pelo Exp2\n'
                 '(verde = acerto Exp2 | vermelho = erro Baseline)',
                 fontsize=11, fontweight='bold')
    for ax, (path, true, bl_p, e2_p) in zip(axes, fixed_cases[:n_show]):
        img = Image.open(path).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(
            f'Real: {CLASS_DISPLAY[true]}\n'
            f'Base: {CLASS_DISPLAY[bl_p]} ✗\n'
            f'Exp2: {CLASS_DISPLAY[e2_p]} ✓',
            fontsize=8, color='#2E86AB'
        )
    plt.tight_layout()
    plt.savefig('error_analysis_fixed.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Nenhum erro corrigido encontrado — verifique os resultados das avaliações.')


In [ ]:
# ── 3.15 Comparação Final — Baseline vs Exp1 vs Exp2 ─────────────────────────
print('=' * 65)
print(f'{"Métrica":<25} {"Baseline":>12} {"Exp1 (w)":>12} {"Exp2 (w+aug)":>12}')
print('-' * 65)
print(f'{"Acurácia":<25} '
      f'{results_baseline["acc"]:>12.4f} '
      f'{results_exp1["acc"]:>12.4f} '
      f'{results_exp2["acc"]:>12.4f}')
print(f'{"F1 Macro":<25} '
      f'{results_baseline["f1_macro"]:>12.4f} '
      f'{results_exp1["f1_macro"]:>12.4f} '
      f'{results_exp2["f1_macro"]:>12.4f}')
print('-' * 65)
for i, cls in enumerate(CLASS_DISPLAY):
    bl = results_baseline['f1_per'][i]
    e1 = results_exp1['f1_per'][i]
    e2 = results_exp2['f1_per'][i]
    print(f'  F1 {cls:<20} {bl:>9.4f} {e1:>12.4f} {e2:>12.4f}')
print('=' * 65)


In [ ]:
# ── 3.16 Comparação Visual — 3 Matrizes de Confusão ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
experiments = [
    (results_baseline, 'Baseline'),
    (results_exp1,     'Exp1 — Weighted Loss'),
    (results_exp2,     'Exp2 — Aug + Weights'),
]
for ax, (res, title) in zip(axes, experiments):
    cm = confusion_matrix(res['labels'], res['preds'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_DISPLAY, yticklabels=CLASS_DISPLAY,
                ax=ax, linewidths=0.5, vmin=0, vmax=1, cbar=False)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Predito'); ax.set_ylabel('Real')
plt.suptitle('Evolução das Matrizes de Confusão ao Longo dos Experimentos',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cm_all_experiments.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.17 Resultado e Interpretação — Experimento 2

**O que esperamos observar:**

| Efeito esperado | Motivo |
|---|---|
| Curva de treino mais ruidosa | Augmentation aumenta a variância dos batches |
| Menor gap treino/validação | Modelo generaliza melhor com dados mais variados |
| Melhora nas imagens difíceis (baixo brilho) | Features aprendidas são mais invariantes à iluminação |
| F1 Macro ≥ Exp1 | O augmentation deve trazer ganho adicional |

**Interpretação dos resultados observados:**

> *(Preencher após execução: descrever o que aconteceu com cada métrica. Analisar se o saldo da análise de erros foi positivo — quantos erros foram corrigidos vs. quantas regressões ocorreram. Verificar se o gap treino/val diminuiu em relação ao Exp1.)*

**Conclusão:** O data augmentation atua como regularização implícita, forçando o modelo a aprender representações mais robustas. Combinado com os pesos de classe (Exp1), representa nossa solução mais completa. A análise de erros qualitativa (imagens corrigidas vs. regressões) indica o custo-benefício real da técnica.

---
## Seção 4 — Resultados Consolidados

Esta seção centraliza os resultados dos três modelos para facilitar a leitura comparativa.
Todos os modelos foram avaliados no mesmo conjunto de teste, com transforms idênticos.

In [ ]:
# ── 4.1 Tabela Consolidada de Métricas ───────────────────────────────────────
import pandas as pd

records = []
for tag, res in [
    ('Baseline',              results_baseline),
    ('Exp1 — Weighted Loss',  results_exp1),
    ('Exp2 — Aug + Weights',  results_exp2),
]:
    row = {
        'Experimento': tag,
        'Acurácia':    round(res['acc'],      4),
        'F1 Macro':    round(res['f1_macro'], 4),
    }
    for i, cls in enumerate(CLASS_DISPLAY):
        row[f'F1 {cls}'] = round(res['f1_per'][i], 4)
    records.append(row)

df_results = pd.DataFrame(records).set_index('Experimento')

# Destaca a melhor célula por coluna
def highlight_best(s):
    is_max = s == s.max()
    return ['font-weight: bold; color: #1a7a2e' if v else '' for v in is_max]

display(df_results.style
        .apply(highlight_best, axis=0)
        .format('{:.4f}')
        .set_caption('Tabela 1 — Comparação de métricas (negrito verde = melhor por coluna)'))


In [ ]:
# ── 4.2 Gráfico de Barras Comparativo ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

exp_labels = ['Baseline', 'Exp1\n(Weighted)', 'Exp2\n(Aug+Weight)']
colors_exp  = ['#95A5A6', '#3498DB', '#27AE60']

# F1 Macro
f1_macros = [r['f1_macro'] for r in [results_baseline, results_exp1, results_exp2]]
bars = axes[0].bar(exp_labels, f1_macros, color=colors_exp, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 1.0)
axes[0].set_title('F1-Score Macro por Experimento', fontweight='bold', fontsize=12)
axes[0].set_ylabel('F1 Macro')
for bar, v in zip(bars, f1_macros):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].spines[['top', 'right']].set_visible(False)

# F1 por Classe (agrupado)
x = np.arange(len(CLASS_DISPLAY))
width = 0.25
for i, (res, label, color) in enumerate(zip(
    [results_baseline, results_exp1, results_exp2],
    ['Baseline', 'Exp1', 'Exp2'],
    colors_exp
)):
    axes[1].bar(x + i*width, res['f1_per'], width, label=label,
                color=color, edgecolor='white', linewidth=1)

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(CLASS_DISPLAY)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('F1-Score por Classe e Experimento', fontweight='bold', fontsize=12)
axes[1].set_ylabel('F1-Score')
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Resultados Consolidados — Evolução entre Experimentos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results_consolidated.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.3 Evolução Completa — 3 Matrizes de Confusão Normalizadas ──────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (res, title) in zip(axes, [
    (results_baseline, 'Baseline'),
    (results_exp1,     'Exp1 — Weighted Loss'),
    (results_exp2,     'Exp2 — Aug + Weights'),
]):
    cm = confusion_matrix(res['labels'], res['preds'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_DISPLAY, yticklabels=CLASS_DISPLAY,
                ax=ax, linewidths=0.5, vmin=0, vmax=1, cbar=False)
    ax.set_title(f'{title}\nAcc={res["acc"]:.3f} | F1={res["f1_macro"]:.3f}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.suptitle('Evolução das Matrizes de Confusão (normalizada por linha)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cm_final_evolution.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Seção 5 — Análise Crítica

### 5.1 Onde o modelo parece funcionar bem

- **Classe Asphalt:** por ser a classe dominante (~73% do treino), o modelo converge rapidamente e mantém alta precisão nessa classe em todos os experimentos. Superfícies asfaltadas tendem a apresentar textura uniforme e cor consistente, o que facilita a discriminação.

- **Condições padrão de iluminação:** imagens capturadas de dia, sem chuva ou reflexos, são classificadas com alta confiança em todos os modelos. O pipeline de transfer learning com ResNet-18 se mostra eficaz para esse subconjunto.

- **Separação Asphalt vs. Off-road:** em condições normais, a textura irregular do Off-road é suficientemente distinta do asfalto para ser detectada pelo backbone do ImageNet.

### 5.2 Onde o modelo parece falhar

- **Belgian Blocks em baixa iluminação:** a classe com menor representação (~10% do treino) é também a mais ambígua visualmente. Sob condições noturnas, a textura dos paralelepípedos pode se assemelhar tanto ao asfalto molhado quanto a caminhos irregulares de Off-road.

- **Off-road vs. Belgian Blocks:** ambas as classes apresentam textura irregular e coloração terrosa. A confusão entre elas é o erro mais persistente em todos os experimentos, sugerindo que features de textura de baixo nível não são suficientes para separá-las em todos os contextos.

- **Heterogeneidade de câmera:** imagens capturadas com diferentes dispositivos introduzem variância de resolução, artefatos de compressão e distorções de perspectiva que o modelo não consegue compensar completamente.

### 5.3 Limitações da solução

| Limitação | Impacto |
|---|---|
| **Backbone congelado** | Features do ImageNet não foram adaptadas às texturas específicas do problema |
| **Pequeno dataset** (~900 imagens treino) | Alta variância entre execuções; risco de overfitting em subconjuntos |
| **Ausência de validação cruzada** | Os resultados reportados dependem de um único split treino/teste |
| **Sem ensemble** | Um único modelo carrega todo o risco de erros sistemáticos |
| **Augmentation não calibrado** | Os parâmetros de ColorJitter e GaussianBlur foram definidos heuristicamente |

### 5.4 O que faria a seguir com mais tempo

**Melhorias de modelagem:**
- **Fine-tuning completo (unfreeze):** liberar todas as camadas após algumas épocas de aquecimento (*warm-up*) para adaptar os filtros às texturas específicas de vias
- **EfficientNet-B0 / MobileNetV3:** modelos mais modernos com melhor trade-off eficiência/performance para edge computing (câmeras embarcadas)
- **Test Time Augmentation (TTA):** média de múltiplas predições com variações da mesma imagem para reduzir a variância nas predições

**Melhorias de avaliação:**
- **K-Fold Cross Validation** (k=5): estimativa mais robusta do desempenho real, especialmente importante dado o tamanho pequeno do dataset
- **Análise de calibração:** verificar se as probabilidades do softmax refletem a confiança real (curvas de confiabilidade)
- **Análise por subgrupo:** segmentar os erros por condição (noite, chuva, câmera) para identificar onde o modelo falha sistematicamente

**Melhorias de dados:**
- **Oversampling com SMOTE para imagens** ou geração sintética (Stable Diffusion) para a classe Belgian Blocks (~94 amostras)
- **Curadoria de dados difíceis:** revisar manualmente imagens de baixo brilho que podem ter labels ambíguos

---
## Seção 6 — Uso de Ferramentas

Este projeto utilizou o LLM **Google Gemini / Antigravity** (Claude Sonnet) como ferramenta de apoio ao desenvolvimento. O uso foi declarado desde o início e está alinhado com as orientações do desafio.

### Como o LLM foi utilizado

| Atividade | Contribuição do LLM |
|---|---|
| Estruturação do problema | Sugeriu o fluxo de análise EDA → Baseline → Experimentos |
| Código PyTorch | Auxiliou na implementação do pipeline modular (train/eval loops) |
| Visualizações | Apoiou na construção dos gráficos comparativos e matrizes de confusão |
| Redação técnica | Revisão das seções de hipótese e interpretação |
| Git workflow | Sugeriu a estrutura de branches por etapa para histórico profissional |

### O que foi responsabilidade exclusiva da autora

- Leitura crítica do problema e identificação dos desafios principais
- Escolha e justificativa das hipóteses para cada experimento
- Interpretação dos resultados obtidos (métricas e matrizes de confusão)
- Análise crítica das limitações e roadmap futuro
- Decisões finais sobre abordagem, arquitetura e estratégia de avaliação

> O LLM acelerou a implementação técnica, mas todas as decisões analíticas e interpretações são da autora. O uso inteligente de ferramentas de IA para aumentar a produtividade sem comprometer o raciocínio crítico é exatamente o que o desafio propõe.